In [10]:
import json
data_list = []

with open('data/qa_pairs.jsonl', 'r', encoding="utf-8") as file:
    for line in file:
        data_list.append(json.loads(line))

for i in range(5):
    print("question: ", data_list[i]['question'])
    print("context ", data_list[i]['context'])

question:  Какие маневры совершили отряды Элфхельма и Гримбольда, когда король задержался у разбитых ворот?
context  Король ненадолго придержал свою свиту у разбитых северных ворот Раммас Экор. За это время первый эоред окружил его с двух сторон и с тыла. Горедар старался держаться как можно ближе к королю, хотя эоред Элфхельма повернул вправо. Всадники Гримбольда свернули влево и поскакали дальше на восток, к большому пролому в стене.
Мерри выглянул из-за спины Горедара. Далеко, милях в десяти отсюда, был виден большой пожар, а между ним и королевским войском линия огня выгнулась полумесяцем, ближний край его был совсем рядом, на расстоянии гона. Хоббит мало что понял, ничего больше на темном поле не рассмотрел и, как и раньше, не чувствовал никакого прояснения или свежего ветра и не надеялся на наступление утра.
Преодолев наружную стену, всадники въехали на поля Пеленнора медленно и почти бесшумно, но в их рядах чувствовался скрытый напор волны прибоя, который вдруг рвет преграды, ка

In [6]:
# Подготовка датасета
import os
from src.clean.clean_data import DataCleaner
from src.fb2_to_txt import DatasetPrepare
import bs4

DATA_PATH = 'data/dirty/'
SAVE_TXT_PATH = 'data/cleaned/'

for filename in os.listdir(DATA_PATH):
    with open(DATA_PATH + filename, "r", encoding="utf-8") as file:
        text = file.read()

    dc = DataCleaner(text)
    txt = dc.clean_all()

    name_only = os.path.splitext(filename)[0]
    save_path = os.path.join(SAVE_TXT_PATH, name_only + ".txt")
    with open(save_path, "w", encoding="utf-8") as f:
        f.write(txt)

In [ ]:
from unsloth import FastLanguageModel
import torch

# Загрузка модели
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-4B",
    max_seq_length = 2048,
    dtype = torch.bfloat16,
    load_in_4bit = False,  # bf16 LoRA, влезает в 16GB
)


ModuleNotFoundError: No module named 'unsloth'

In [ ]:
# Переключаем в режим инференса (быстрее и меньше памяти)
FastLanguageModel.for_inference(model)

# Промпт

test_prompts = [
    "In a hole in the ground there lived",
    "The Elvish script upon the ring read:",
    "Long ago in the Second Age of Middle-earth,",
    "Gandalf spoke softly but his words carried",
]

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.8)
    print(f"\n--- PROMPT: {prompt} ---")
    print(tokenizer.decode(out[0], skip_special_tokens=True))

In [7]:
from datasets import Dataset

def chunk_text(text, chunk_size=512, overlap=50):
    tokens = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(tokens) - chunk_size, chunk_size - overlap):
        chunk = tokens[i:i + chunk_size]
        chunks.append(tokenizer.decode(chunk))
    return chunks

# Читаем все очищенные файлы
all_chunks = []
for filename in os.listdir("data/cleaned/"):
    with open(f"data/cleaned/{filename}", "r", encoding="utf-8") as f:
        text = f.read()
    all_chunks.extend(chunk_text(text))

dataset = Dataset.from_dict({"text": all_chunks})
print(f"Всего чанков: {len(dataset)}")

ModuleNotFoundError: No module named 'datasets'

In [ ]:
FastLanguageModel.for_training(model)

# Навешиваем LoRA адаптеры
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=10,
        output_dir="./tolkien_lora",
        save_strategy="epoch",
    ),
)
trainer.train()

# Сохраняем адаптер
model.save_pretrained("tolkien_lora_adapter")

In [ ]:
FastLanguageModel.for_inference(model)

# Те же test_prompts — сравниваешь с baseline
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.8)
    print(f"\n--- PROMPT: {prompt} ---")
    print(tokenizer.decode(out[0], skip_special_tokens=True))